# 1 Test SR model

### 1.1 First we can test the model on a single image to verify that it works correctly. We will load a test image, run it through the model, and visualize the output.

In [ ]:
import numpy as np 
import torch
from PIL import Image
import matplotlib.pyplot as plt

from src.emcfsys.EMCellFiner.hat.models.hat_model import HATModel
from src.emcfsys.EMCellFiner.hat.models.img_utils import tensor2img

# 1. init the model
local_path = r"D:\napari_EMCF\EMCFsys\models\EMCellFiner.pth"
# set the tile_size according to your GPU memory, 512 is recommended for 8GB GPU
model = HATModel(scale=4, tile_size=512) 
# model = HATModel(scale=4, tile_size=512, local_path=local_path)  # for 8GB GPU, use half precision to save memory
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# 2. read the image
img_path = r"D:\napari_EMCF\EMCFsys\emcfsys\tests\test_img.tif"
img = Image.open(img_path).convert("RGB").resize((512, 512), resample=Image.BICUBIC)

# convert to tensor and normalize to [0, 1]
img_np = np.array(img).astype(np.float32) / 255.
# 转换为 Tensor: (H, W, C) -> (C, H, W) -> (1, C, H, W)
img_torch = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).to(device)
print(f"Input Shape: {img_torch.shape}")
# 3. inference

import time
start_time = time.time()
with torch.no_grad():
    output = model(img_torch) 
end_time = time.time()
print(f"Inference Time: {end_time - start_time:.2f} seconds")
print(f"Output Shape: {output.shape}")

# 4. post-process the output tensor
output = output.cpu()
img_out = tensor2img(output, rgb2bgr=False, min_max=(0, 1))
# 5. convert to PIL Image
img_final = Image.fromarray(img_out)

# 5. show and save the image
img_final.show() 
img_final.save("result_sr.png")
print("Done.")

#### 1.2 Then we can measure FLOPs, parameters, memory usage, and inference time

In [ ]:
import numpy as np 
import torch
import matplotlib.pyplot as plt
from src.emcfsys.utils.fps import count_parameters, compute_flops, measure_memory, measure_inference_time
from src.emcfsys.EMCellFiner.hat.models.hat_model import HATModel
from src.emcfsys.EMCellFiner.hat.models.img_utils import tensor2img

# use the same model and input tensor as before
# img_torch = torch.randn(1, 3, 512, 512).to(device)  
count_parameters(model)
compute_flops(model, img_torch)
measure_memory(model, img_torch)

# inference 10 times and average the time
measure_inference_time(model, img_torch)
